<a href="https://colab.research.google.com/github/JsebastianM2002/An-lisis-de-intervalos-de-calibraci-n-equipos-metrol-gicos/blob/main/An%C3%A1lisis_metrol%C3%B3gico_utilizando_regla_de_tres_inversa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files


# Cargar archivo

uploaded = files.upload()

nombre_archivo = list(uploaded.keys())[0]

if nombre_archivo.endswith(".csv"):
    data = pd.read_csv(nombre_archivo)
else:
    data = pd.read_excel(nombre_archivo)

# Extraer datos

valor_patron = data["patron"].values
valor_medido = data["medido"].values
incertidumbre = data["incertidumbre"].values


# Datos adicionales

t = float(input("Tiempo intervalo calibración (meses): "))

tolerancia = float(input("Ingrese el valor EMP del equipo (%): "))
EMP = float(input("Ingrese el EMP de frecuencia (%): "))
EMP = EMP / 100


# Calculos metrologicos


# error absoluto
error_abs = valor_medido - valor_patron

# error porcentual
error_porcentual = np.where(valor_patron != 0, ((valor_medido - valor_patron) / valor_patron) * 100, np.nan)

# incertidumbre porcentual
incertidumbre_porcentual = np.where(valor_patron != 0, (incertidumbre / valor_patron) * 100, np.nan)

# limites error considerando incertidumbre
errormax = error_porcentual + incertidumbre_porcentual
errormax1 = error_porcentual - incertidumbre_porcentual

# error maximo del punto
mayor = np.maximum(np.abs(errormax), np.abs(errormax1))


# Intervalo de calibracion


intervalo = ((tolerancia * EMP * t) / mayor)

intervalo_final = np.nanmin(intervalo)

# Grafica error porcentual respecto al numero de puntos

valor_medidox = np.arange(1, len(error_porcentual)+1)

plt.figure(figsize=(9,6))

plt.errorbar(valor_medidox, error_porcentual, yerr=np.abs(incertidumbre_porcentual),
             fmt='o', capsize=5, color='blue', label="Error máximo")


# tendencia lineal
coef = np.polyfit(valor_medidox[~np.isnan(error_porcentual)], mayor[~np.isnan(error_porcentual)], 1)
tendencia = np.poly1d(coef)

plt.plot(valor_medidox, tendencia(valor_medidox),
         color='green', linewidth=2, label="Tendencia")

# lineas EMP
plt.axhline(tolerancia, color='red', linestyle='--', label="EMP")
plt.axhline(-tolerancia, color='red', linestyle='--')

# lineas 80% EMP
plt.axhline(EMP*tolerancia, color='c', linestyle='--', label=f"{EMP*100:.0f}% EMP")
plt.axhline(-EMP*tolerancia, color='c', linestyle='--')

plt.xlabel("Puntos de medición")
plt.ylabel("Error (%)")
plt.title("Evaluación metrológica del instrumento")

plt.grid(True)
plt.legend()

plt.show()


# Tabla de resultados

tabla = pd.DataFrame({
    "Patron": valor_patron,
    "Medido": valor_medido,
    "Error": error_abs,
    "Error %": error_porcentual,
    "Incertidumbre %": incertidumbre_porcentual,
    "Error maximo": mayor,
    "Intervalo (meses)": intervalo
})

print("\nTabla de resultados:")
print(tabla)
print("Intervalo minimo de frecuencia", intervalo_final, "Meses")